In [1]:
import findspark
findspark.init()

In [2]:
from pyspark import SparkContext
from pyspark.sql import SparkSession, SQLContext
from pyspark.mllib.recommendation import ALS
import os
import shutil

In [3]:
sc = SparkContext()
sql_content = SQLContext(sc)

In [4]:
read = sql_content.read
df = read.schema('user_id Int, book_id Int, hit_num Int').csv(path='./data/hit.csv')
df.registerTempTable('hit')

In [5]:
sqlContext = SparkSession.builder.getOrCreate()
data = sqlContext.sql('select user_id,book_id,sum(hit_num) as hits from hit group by user_id, book_id')
data.show()

+-------+-------+----+
|user_id|book_id|hits|
+-------+-------+----+
|      1|   9464|   1|
|      1|     20|   1|
|      1|      2|   1|
|      1|   9477|   1|
|      1|    446|   1|
|      2|    693|   1|
|      1|      1|   4|
|      1|      3|   1|
|      1|   9468|   1|
|      1|   9462|   1|
|      1|      4|   1|
|      1|      8|   1|
|      1|     16|   1|
|      1|   9645|   1|
|      1|     17|   1|
|      1|     10|   1|
+-------+-------+----+



In [6]:
bookrdd = data.rdd.map(lambda x: (x.user_id, x.book_id, x.hits))
bookrdd.collect()

[(1, 9464, 1),
 (1, 20, 1),
 (1, 2, 1),
 (1, 9477, 1),
 (1, 446, 1),
 (2, 693, 1),
 (1, 1, 4),
 (1, 3, 1),
 (1, 9468, 1),
 (1, 9462, 1),
 (1, 4, 1),
 (1, 8, 1),
 (1, 16, 1),
 (1, 9645, 1),
 (1, 17, 1),
 (1, 10, 1)]

In [ ]:
model = ALS.trainImplicit(bookrdd, 10, 10, 0.01)

In [ ]:
if os.path.exists('recommemdModel'):
    print('delete')
    shutil.rmtree('recommemdModel')

In [ ]:
model.save(sc, 'recommemdModel')
sc.stop()